# ReAct Agent - From Scratch

## Load Keys from Env.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from cohere import ClientV2

co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

In [3]:
MODEL_NAME = "command-a-03-2025"

## Define Tools And Tool Registry

In [4]:
from datetime import datetime, date, timedelta


def calculator(expression):
    safe_globals = {
        "__builtins__": {},
        "datetime": datetime,
        "date": date,
        "timedelta": timedelta,
        "abs": abs,
        "min": min,
        "max": max,
        "round": round,
    }
    return {
        "result": eval(
            expression,
            safe_globals,
            {},
        )
    }


##########################################

KB = {
    "price_per_ticket": 89,
    "attendees": 47,
    "conference_date": "2026-09-14",
    "venue": "Lisbon Congress Center",
}


def lookup(key):
    return {
        "value": KB.get(key, f"unknown key: {key}"),
        "available_keys": list(KB.keys()),
    }


def get_today():
    return {"today": date.today().isoformat()}


TOOL_REGISTRY = {
    "calculator": calculator,
    "lookup": lookup,
    "get_today": get_today,
}

## Describe Tools

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluates a Python expression and returns the result. "
                "Supports arithmetic (e.g. '23 * 47') AND date math via date.fromisoformat('YYYY-MM-DD'). "
                "To compute days between two dates: "
                "\"(date.fromisoformat('2026-09-14') - date.fromisoformat('2026-05-14')).days\". "
                "Do NOT pass raw quoted date strings like \"'2026-09-14' - '2026-05-14'\" — that's string subtraction and will fail."
                "ONLY pass date expressions or arithmetic expressions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "e.g. '23 * 47' or \"(date.fromisoformat('2026-09-14') - date.fromisoformat('2026-05-14')).days\"",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup",
            "description": "Look up a fact in the conference knowledge base by key (e.g. 'attendees', 'price_per_ticket', 'conference_date', 'venue').",
            "parameters": {
                "type": "object",
                "properties": {
                    "key": {"type": "string", "description": "e.g. 'attendees'"}
                },
                "required": ["key"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_today",
            "description": "Returns today's date in ISO format (YYYY-MM-DD). Use this whenever the question depends on knowing the current date.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

## Agent Loop

In [6]:
import json


def run_agent(user_question, max_steps=8):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise assistant. Use the tools to gather facts and to do any arithmetic. "
                "When you have enough information, stop calling tools and answer."
                "When calling the calculator tool, ALWAYS pass a single-line expression with no "
                "comments, no newlines, and no variable assignments."
            ),
        },
        {
            "role": "user",
            "content": user_question,
        },
    ]

    for step in range(1, max_steps + 1):
        resp = co.chat(model=MODEL_NAME, messages=messages, tools=tools)

        if not resp.message.tool_calls:
            print(f"[step {step}] final answer")
            return resp.message.content[0].text

        print(f"[step {step}] plan: {resp.message.tool_plan}")
        messages.append(
            {
                "role": "assistant",
                "tool_calls": resp.message.tool_calls,
                "tool_plan": resp.message.tool_plan,
            }
        )

        for tc in resp.message.tool_calls:
            args = json.loads(tc.function.arguments)
            result = TOOL_REGISTRY[tc.function.name](**args)
            print(f"        → {tc.function.name}({args}) = {result}")
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result),
                }
            )

    return "⚠️ Hit max steps without producing a final answer."

## User Input

In [7]:
question = (
    "What's the total ticket revenue for the conference, when and where is it happening, "
    "and how many days from today until it starts?"
)

answer = run_agent(question)
print("\n=== FINAL ANSWER ===\n", answer)

[step 1] plan: I will use the 'lookup' tool to find the total ticket revenue for the conference, the date of the conference, and the venue. I will then use the 'get_today' tool to find out today's date. Finally, I will use the 'calculator' tool to find out how many days there are between today and the conference.
        → lookup({'key': 'price_per_ticket'}) = {'value': 89, 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        → lookup({'key': 'attendees'}) = {'value': 47, 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        → lookup({'key': 'conference_date'}) = {'value': '2026-09-14', 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        → lookup({'key': 'venue'}) = {'value': 'Lisbon Congress Center', 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        → get_today({}) = {'today': '2026-06-07'}
[step 2] plan: I have found that the price per tic

> Start with the simplest thing that works.
> 
> If a single augmented LLM call solves the task, ship it.
> 
>  If not, design a workflow.
> 
> Only reach for an agent when the steps themselves depend on the input in a way you can't predict.
